# `ssa_inverted_solve_budd`

> Generated from `config/ais_0.1_SSA.py` by `config/make_step_notebooks.py` on 2026-10-28. Edit the driver and regenerate; changes made here are overwritten.

Forward SSA solve with the Budd p=q=1 friction field (SSA track start).

**Reads** (`load_model` calls in this step):
- `f'{model_dir}/AIS3_param.nc'`
- `f'{FRICTION_RUN_DIR}/{FRICTION_RUN_NAME}.nc'`

**Writes** (`save_model` calls in this step):
- `f'{model_dir}/AIS3_SSA_inverted.nc'`

**Run as a batch job** (from `config/notebooks/`; resources copied from `config/launch_ais_0.1_SSA.pbs`):

```bash
qsub -q normal -l ncpus=8,mem=190GB,walltime=25:00:00 -v NB=ais_0.1_SSA/00_ssa_inverted_solve_budd.ipynb launch_notebook.pbs
```

The executed copy is written next to this notebook as `executed/<name>.<jobid>.ipynb`. Interactive use needs a Jupyter session with at least those resources.

**Run options**: none.

Status, results and known failures: [`docs/run_guide.md`](../../../docs/run_guide.md) and the worklogs in `docs/`.

## Setup

The preamble of `ais_0.1_SSA.py`: imports, helpers, settings and the data catalog.

In [ ]:
"""ACCESS-AIS3 pure-SSA production pipeline (Budd/Weertman friction law, p=q=1 -- the best
validated RMSE anywhere in this project, 60.4 m/yr, vs. the Schoof-based HO track's current
94-99 m/yr after several chunked-restart iterations). See docs/inversion_worklog.md for the
full history of how p=q=1 was found, and the "in your opinion" exchange in this session's
transcript for why a dedicated SSA production track is worth having alongside the HO one:
SSA is dramatically cheaper and already converged/stable, at the cost of no vertical-shear
physics (so it can't do thermal-coupled work the way the HO track can).

Deliberately self-contained -- does NOT import or exec ais_0.1.py, and does NOT touch its
shared `steps`/`save` globals. This project hit two real race-condition incidents this
session from toggling those globals while a job was mid-flight; a fully separate file with
its own local `steps`/`save` sidesteps that risk entirely, at the cost of some code
duplication (the handful of helper functions and Budd-specific step logic below are copied
from ais_0.1.py, not imported, so the two files can diverge safely and be run concurrently).

Friction source: models/AIS3_ssa_friction_inv_reg_lcurve/run_001_10_100_0.0001/ -- the real,
production-canonical location the RMSE=60.4 p=q=1 result was salvaged into (see
config/finalize_p1q1_reg_lcurve.py) from the ad-hoc script that originally produced it
(p1q1_reg_sweep2.py, no longer present as a file, only as raw execution output). This
finalization step recovers the ACTUAL computed result rather than re-deriving it from
scratch, which would otherwise require regenerating AIS3_param.nc with Budd config (it
currently has Schoof baked in, for the HO track) and re-running two expensive inversion
stages. Run finalize_p1q1_reg_lcurve.py once before this script's first stage.

Stage status (updated 2026-09-02):
  1. ssa_inverted_solve_budd -- FULLY IMPLEMENTED AND RUN (RMSE=65.67 m/yr full-continental,
     the best full-continental result in the project). AIS3_SSA_inverted.nc.
  2. ssa_relaxation_budd -- FULLY IMPLEMENTED AND RUN (mean |dH|=32.90m, max |dH|=1505.39m
     over the 20yr relaxation). AIS3_SSA_relaxed.nc.
  3. melt_gamma_tuning_ssa -- builds AIS3_melt_final_ssa.nc directly from the validated
     calibration (gamma_0=300 + per-basin refit deltaT_basin, hard-coded; 2026-09-29). The
     values come from the standalone refit chain (ssa_melt_deltaT_basin_refit.py /
     ssa_melt_deltaT_refit_recalibrate.py). The existing AIS3_melt_final_ssa.nc was built by
     finalize_ssa_melt_calibration.py. The step's old coarse gamma_0 sweep (pick 5537.7,
     AIS3_melt_gamma_tuning_ssa.nc) was superseded by that refit and removed.
  4. historical_dhdt_tuning_ssa -- RUN AND VALIDATED. Full 1995-2019 (24yr) transient against
     AIS3_melt_final_ssa.nc, checked against MIPKIT's dhdt_cpom: grounded RMSE=1.402 m/yr,
     independently reconfirmed via check_historical_ssa.py. AIS3_historical_1995_2019_SSA.nc.
  5. projection_ssp_ssa -- IMPLEMENTED (2026-09-29): anomaly forcing on the calibrated
     baseline (TF0 = Zhou, SMB0 = RACMO 1995-2014), CTRL/SSP/attribution/melt variants chosen
     per job via qsub -v, self-chaining 50-yr chunks to 2300. See launch_projection_ssa.pbs and
     docs/projection_experiments.md.

Both new stages use the plain `cluster` (48 cores/190GB/normal) config, not hugemem -- the
SSA mesh is ~15x smaller than the HO track's (no vertical layers), and both stages use the
same synchronous submit-and-wait pattern (`waitonlock` minutes + `load_only=False` in one
call) already validated throughout this pipeline, avoiding the "never actually submits" bug
found and fixed in ais_0.1.py's own ho_relaxation/historical_dhdt_tuning on their first real
run.
"""
import pyissm
import ccdtools as ccdtools
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import xarray as xr
import os
import glob
from pathlib import Path
from scipy.spatial import Delaunay

os.chdir('/g/data/au88/jh7060/ACCESS-AIS3/')
os.environ['ISSM_DIR'] = '/g/data/vk83/apps/spack/1.1/release/linux-x86_64/issm-git.2026.05.18_2026.05.18-kgta35igm37z4qnqnul7rcmgx2inftqd'

model_dir = '/g/data/au88/jh7060/ACCESS-AIS3/models'
execution_dir = '/g/data/au88/jh7060/ACCESS-AIS3/execution_SSA'

save = True
diagnostics = True
plot = True

cluster = pyissm.model.classes.cluster.gadi()
cluster.codepath = os.environ['ISSM_DIR'] + '/bin'
cluster.executionpath = execution_dir
cluster.storage = 'gdata/au88+gdata/vk83+gdata/av17'
cluster.moduleuse = ['/g/data/vk83/modules/']
cluster.moduleload = ['access-issm_ad/2026.05.0']
cluster.np = 48
cluster.memory = 190
cluster.time = 60 * 48
cluster.login = 'jh7060'
cluster.project = 'au88'

FRICTION_RUN_DIR = f'{model_dir}/AIS3_ssa_friction_inv_reg_lcurve/run_001_10_100_0.0001'
FRICTION_RUN_NAME = 'run_001_10_100_0.0001'

all_steps = ['ssa_inverted_solve_budd', 'ssa_relaxation_budd', 'melt_gamma_tuning_ssa', 'historical_dhdt_tuning_ssa', 'projection_ssp_ssa']
steps = ['historical_dhdt_tuning_ssa']  # edit before each qsub -- see launch_ais_0.1_SSA.pbs
# Per-job override via `qsub -v AIS3_STEPS=a,b` (same as ais_0.1.py) -- lets a launcher pick its
# step without editing the shared `steps` above.
if os.environ.get('AIS3_STEPS'):
    steps = os.environ['AIS3_STEPS'].split(',')


def _interp_weights(src_x, src_y, tgt_x, tgt_y):
    """Precompute a reusable linear-interpolation transform (Delaunay triangulation +
    barycentric weights) ONCE for a fixed (source-grid, target-mesh) pair, so many
    different value arrays defined on that same source grid can each be interpolated in
    milliseconds afterward via _interp_apply.

    Needed for projection_ssp_ssa's time-varying ocean thermal forcing: 30 depths x ~81
    years means ~2430 interpolations of the SAME 761x761 source grid onto the SAME mesh --
    pyissm.data.interp.points_to_mesh (scipy.interpolate.griddata under the hood) rebuilds
    the full Delaunay triangulation AND redoes point-location for every single call, which
    is fine for this pipeline's existing handful-of-calls loops (RACMO's 24 annual SMB
    calls) but empirically verified to cost ~13-20s PER CALL even with triangulation reuse
    alone (evaluation/point-location dominates, not triangulation) -- at 2430 calls that is
    hours just for TF preprocessing, dwarfing the actual transient solve. Precomputing the
    per-target-point simplex + barycentric weights ONCE instead drops each subsequent
    interpolation to ~10ms (verified empirically), since applying already-known weights to
    a new value array is just a weighted sum, not a fresh geometric search.
    """
    src_xy = np.column_stack([np.asarray(src_x).ravel(), np.asarray(src_y).ravel()])
    tgt_xy = np.column_stack([np.asarray(tgt_x).ravel(), np.asarray(tgt_y).ravel()])
    tri = Delaunay(src_xy)
    simplex = tri.find_simplex(tgt_xy)
    vertices = np.take(tri.simplices, simplex, axis=0)
    temp = np.take(tri.transform, simplex, axis=0)
    delta = tgt_xy - temp[:, 2]
    bary = np.einsum('njk,nk->nj', temp[:, :2, :], delta)
    wts = np.hstack((bary, 1 - bary.sum(axis=1, keepdims=True)))
    return vertices, wts, simplex


def _interp_apply(values, vtx, wts, simplex, fill_value=np.nan):
    """Apply a transform from _interp_weights to a new value array defined on the same
    source grid it was built from -- same interpolation this pipeline uses everywhere else
    (linear, NaN outside the source grid's convex hull), just without repeating the
    triangulation/point-location work each time."""
    values = np.asarray(values).ravel()
    out = np.einsum('nj,nj->n', values[vtx], wts)
    out[simplex == -1] = fill_value
    return out


def load_shelf_rheology_B():
    """Identical to ais_0.1.py's own helper -- copied, not imported, deliberately (see
    module docstring). The validated floating-shelf rheology result, independent of
    friction law entirely."""
    _cl = pyissm.model.classes.cluster.gadi()
    _cl.codepath = os.environ['ISSM_DIR'] + '/bin'
    _cl.executionpath = '/g/data/au88/jh7060/ACCESS-AIS3/execution_newB_rheology'
    _cl.login = 'jh7060'; _cl.project = 'au88'; _cl.storage = 'gdata/au88'
    mshelf = pyissm.model.io.load_model(f'{model_dir}/AIS3_param.nc')
    mshelf.mask.ice_levelset = pyissm.model.param.kill_icebergs(mshelf)
    sel = (mshelf.mask.ocean_levelset < 0) & (mshelf.mask.ice_levelset < 0)
    mshelf = mshelf.extract(sel)
    mshelf.cluster = _cl
    mshelf.settings.waitonlock = 0
    mshelf.inversion.iscontrol = 0
    mshelf.miscellaneous.name = 'run_001_1_10_1e-17'
    mr = pyissm.model.execute.solve(mshelf, 'Stressbalance', load_only=True, runtime_name=False, check_consistency=False)
    return np.asarray(mr.mesh.extractedvertices).ravel(), np.asarray(mr.results.StressbalanceSolution.MaterialsRheologyBbar).ravel()

## Step `ssa_inverted_solve_budd`

In [ ]:
steps = ['ssa_inverted_solve_budd']  # this notebook's step; overrides the driver default and AIS3_STEPS

In [ ]:
print("-------------------------------------------------------------")
print(" ASSEMBLING PURE-SSA BUDD (p=q=1) INVERTED MODEL")

In [ ]:
print("-------------------------------------------------------------")

In [ ]:
print("-- Loading parameterized model...")
md = pyissm.model.io.load_model(f'{model_dir}/AIS3_param.nc')

In [ ]:
# AIS3_param.nc currently has friction_law='schoof' baked in (built for the HO track) --
# explicitly force Budd/p=q=1 here rather than trusting whatever class it already has.
print("-- Forcing Budd/default friction class, p=q=1...")
md.friction = pyissm.model.classes.friction.default(md.friction)
md.friction.p = np.full(md.mesh.numberofelements, 1.0)
md.friction.q = np.full(md.mesh.numberofelements, 1.0)
md.friction.coefficient = np.full(md.mesh.numberofvertices, 1.8)

In [ ]:
print(f"-- Updating rheology B from floating-ice rheology L-curve...")
_ev, _Bshelf = load_shelf_rheology_B()
md.materials.rheology_B[_ev - 1] = _Bshelf

In [ ]:
print(f"-- Grafting friction field from {FRICTION_RUN_NAME} (RMSE=60.4 m/yr, see finalize_p1q1_reg_lcurve.py)...")
mdf = pyissm.model.io.load_model(f'{FRICTION_RUN_DIR}/{FRICTION_RUN_NAME}.nc')
_fld = np.asarray(md.friction.coefficient).astype(float)
_grafted = np.asarray(mdf.results.StressbalanceSolution.FrictionCoefficient).ravel()
_fld[mdf.mesh.extractedvertices - 1] = _grafted
md.friction.coefficient = _fld

In [ ]:
print('-- Removing icebergs from ice levelset...')
md.mask.ice_levelset = pyissm.model.param.kill_icebergs(md)

In [ ]:
print("-- Flooring thin ice at 100m (numerical stability), preserving observed surface where possible...")
# Identical geometry treatment to ais_0.1.py's ssa_inverted_solve -- must match what
# ssa_friction_inv_reg_lcurve actually solved against (see finalize_p1q1_reg_lcurve.py,
# which applies the same flooring before extracting the friction-inversion domain).
_ri = md.materials.rho_ice; _rw = md.materials.rho_water
_H = np.asarray(md.geometry.thickness).ravel().copy()
_ol = np.asarray(md.mask.ocean_levelset).ravel()
_surf0 = np.asarray(md.geometry.surface).ravel().copy()
_bed = np.asarray(md.geometry.bed).ravel()
_H = np.maximum(_H, 100.0)
_flt = _ol < 0
_base = np.empty_like(_H); _surf = np.empty_like(_H)
_base[_flt] = -_H[_flt] * _ri / _rw; _surf[_flt] = _H[_flt] * (1.0 - _ri / _rw)
_base[_flt] = np.maximum(_base[_flt], _bed[_flt])
_surf[_flt] = _base[_flt] + _H[_flt]
_base[~_flt] = _bed[~_flt]
_surf[~_flt] = _bed[~_flt] + _H[~_flt]
md.geometry.thickness = _H; md.geometry.base = _base; md.geometry.surface = _surf

In [ ]:
print("-- Re-flooring effective pressure against the updated thickness...")
_lim = 0.07
N = md.friction.effective_pressure.copy()
N[N < 0] = 0
_Nfloor = _lim * md.materials.rho_ice * md.constants.g * _H
N = np.maximum(N, _Nfloor)
md.friction.effective_pressure = N
md.friction.effective_pressure_limit = _lim

# NOTE: ais_0.1.py's ssa_inverted_solve has an extensive Schoof-only block here (3 waves
# of Dirichlet pins for thin-ice/lower-bound-C/non-ice vertices, working around Schoof's
# Coulomb-cap instability at the thickness floor -- see docs/inversion_worklog.md 5.4).
# None of that applies to Budd (resistance grows, however weakly, with velocity, so
# Newton always finds a finite equilibrium -- no yield-cap runaway mechanism exists for
# this friction law). Deliberately omitted, not forgotten.

In [ ]:
print("-- Disabling inversion (forward solve only)...")
md.inversion.iscontrol = 0
md.verbose.solution = 1

In [ ]:
print("-- Assigning cluster and updating settings...")
md.miscellaneous.name = 'AIS3_SSA_inverted'
md.cluster = cluster
md.settings.waitonlock = 120  # minutes
md.settings.solver_residue_threshold = 1e-3

if save:
    print("-- Submitting and waiting on stress balance solution...")
    md = pyissm.model.execute.solve(md, 'Stressbalance', load_only=False, runtime_name=False)

    if diagnostics:
        vel = np.asarray(md.results.StressbalanceSolution.Vel).ravel()
        vel_obs = np.asarray(md.inversion.vel_obs).ravel()
        il = np.asarray(md.mask.ice_levelset).ravel()
        ol = np.asarray(md.mask.ocean_levelset).ravel()
        gr = (il < 0) & (ol >= 0) & (vel_obs > 0)
        print(f"\nFORWARD SOLVE DIAGNOSTICS:")
        print(f"   Max modelled velocity: {np.nanmax(vel):.2f} m/yr")
        print(f"   Grounded RMSE vs obs:  {np.sqrt(np.nanmean((vel[gr]-vel_obs[gr])**2)):.2f} m/yr (target: ~60.4)")

    if plot:
        fig, ax, _trip = pyissm.plot.plot_model_field(
            md, md.results.StressbalanceSolution.Vel, show_cbar=True, cmap='PuOr',
            cbar_kwargs={'label': 'Modelled velocity (m/a)'})
        ax.set_title('AIS3_SSA_inverted -- pure-SSA Budd p=q=1 stress balance velocity')
        plt.savefig(f'{model_dir}/AIS3_SSA_inverted_velocity.png')

    print(f"\nSaving to {model_dir}/AIS3_SSA_inverted.nc")
    pyissm.model.io.save_model(md, f'{model_dir}/AIS3_SSA_inverted.nc')
else:
    print("-- Submitting stress balance solve...")
    md = pyissm.model.execute.solve(md, 'Stressbalance', load_only=False, runtime_name=False)